In [ ]:
# ============================================================
# PROJECT 8: HEART DISEASE
# DECISION TREE CLASSIFICATION
# ============================================================

# ------------------------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.tree import DecisionTreeClassifier, plot_tree

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

print("Libraries imported successfully!")


# ------------------------------------------------------------
# 2. UPLOAD DATASET
# ------------------------------------------------------------

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

print("\nDataset loaded successfully!")
print("File:", file_name)


# ------------------------------------------------------------
# 3. BASIC DATA EXPLORATION
# ------------------------------------------------------------

print("\n========== FIRST 5 ROWS ==========")
display(df.head())

print("\n========== LAST 5 ROWS ==========")
display(df.tail())

print("\n========== DATASET SHAPE ==========")
print(df.shape)

print("\n========== COLUMN NAMES ==========")
print(df.columns.tolist())

print("\n========== DATA TYPES ==========")
print(df.dtypes)

print("\n========== DATASET INFORMATION ==========")
df.info()

print("\n========== STATISTICAL SUMMARY ==========")
display(df.describe(include="all").T)

print("\n========== MISSING VALUES ==========")
print(df.isnull().sum())

print("\n========== DUPLICATE ROWS ==========")
print(df.duplicated().sum())


# ------------------------------------------------------------
# 4. CLEAN COLUMN NAMES
# ------------------------------------------------------------

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

print("\nCleaned column names:")
print(df.columns.tolist())


# ------------------------------------------------------------
# 5. HANDLE '?' VALUES
# ------------------------------------------------------------

df = df.replace("?", np.nan)

print("\nMissing values after replacing '?':")
print(df.isnull().sum())


# ------------------------------------------------------------
# 6. REMOVE DUPLICATES
# ------------------------------------------------------------

before = len(df)

df = df.drop_duplicates()

after = len(df)

print("\nDuplicates removed:", before - after)
print("New dataset shape:", df.shape)


# ------------------------------------------------------------
# 7. IDENTIFY TARGET COLUMN
# ------------------------------------------------------------

# Common target names in heart disease datasets

possible_targets = [
    "target",
    "heartdisease",
    "heart_disease",
    "heart_disease_present",
    "output",
    "condition",
    "num",
    "diagnosis"
]

TARGET = None

for column in possible_targets:

    if column in df.columns:

        TARGET = column
        break


# If no target was found automatically,
# change TARGET manually here.

if TARGET is None:

    TARGET = "target"


print("\nTarget column:", TARGET)

if TARGET not in df.columns:

    raise ValueError(
        f"Target column '{TARGET}' was not found.\n\n"
        f"Available columns:\n{df.columns.tolist()}\n\n"
        "Please change the TARGET variable to your actual target column."
    )


# ------------------------------------------------------------
# 8. CONVERT NUMERIC COLUMNS WHERE POSSIBLE
# ------------------------------------------------------------

for column in df.columns:

    if column != TARGET:

        converted = pd.to_numeric(
            df[column],
            errors="coerce"
        )

        # Only replace if conversion is useful
        if converted.notna().sum() >= df[column].notna().sum() * 0.8:

            df[column] = converted


# ------------------------------------------------------------
# 9. TARGET DISTRIBUTION
# ------------------------------------------------------------

print("\n========== TARGET DISTRIBUTION ==========")

print(df[TARGET].value_counts())

print("\nTarget percentages:")

print(
    df[TARGET]
    .value_counts(normalize=True)
    .mul(100)
)


# ------------------------------------------------------------
# 10. TARGET PLOT
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

sns.countplot(
    data=df,
    x=TARGET
)

plt.title("Heart Disease Target Distribution")
plt.xlabel("Heart Disease")
plt.ylabel("Number of Patients")

plt.show()


# ------------------------------------------------------------
# 11. CHECK NUMBER OF CLASSES
# ------------------------------------------------------------

number_of_classes = df[TARGET].nunique()

print(
    "\nNumber of target classes:",
    number_of_classes
)

if number_of_classes != 2:

    print(
        "\nNote: This project is designed for binary "
        "heart disease classification."
    )


# ------------------------------------------------------------
# 12. SEPARATE FEATURES AND TARGET
# ------------------------------------------------------------

X = df.drop(
    columns=[TARGET]
)

y = df[TARGET]


print("\nFeatures shape:", X.shape)
print("Target shape:", y.shape)


# ------------------------------------------------------------
# 13. IDENTIFY NUMERIC FEATURES
# ------------------------------------------------------------

numeric_features = X.select_dtypes(
    include=[
        "int64",
        "float64",
        "int32",
        "float32"
    ]
).columns.tolist()


# ------------------------------------------------------------
# 14. IDENTIFY CATEGORICAL FEATURES
# ------------------------------------------------------------

categorical_features = X.select_dtypes(
    include=[
        "object",
        "category",
        "bool"
    ]
).columns.tolist()


print("\n========== NUMERIC FEATURES ==========")
print(numeric_features)

print("\n========== CATEGORICAL FEATURES ==========")
print(categorical_features)


# ------------------------------------------------------------
# 15. EXPLORATORY DATA ANALYSIS
# ------------------------------------------------------------

if len(numeric_features) > 0:

    X[numeric_features].hist(
        figsize=(16, 12),
        bins=20
    )

    plt.suptitle(
        "Heart Disease Numeric Feature Distributions",
        fontsize=16
    )

    plt.tight_layout()

    plt.show()


# ------------------------------------------------------------
# 16. CORRELATION HEATMAP
# ------------------------------------------------------------

if len(numeric_features) > 1:

    plt.figure(figsize=(12, 9))

    sns.heatmap(
        df[numeric_features].corr(),
        annot=True,
        fmt=".2f",
        cmap="coolwarm"
    )

    plt.title(
        "Correlation Heatmap"
    )

    plt.show()


# ------------------------------------------------------------
# 17. FEATURE VS HEART DISEASE
# ------------------------------------------------------------

for feature in numeric_features[:8]:

    plt.figure(figsize=(8, 5))

    sns.boxplot(
        data=df,
        x=TARGET,
        y=feature
    )

    plt.title(
        f"{feature} vs Heart Disease"
    )

    plt.xlabel("Heart Disease")
    plt.ylabel(feature)

    plt.show()


# ------------------------------------------------------------
# 18. TRAIN-TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


print("\n========== TRAIN-TEST SPLIT ==========")

print(
    "Training samples:",
    X_train.shape[0]
)

print(
    "Testing samples:",
    X_test.shape[0]
)


# ------------------------------------------------------------
# 19. NUMERIC PREPROCESSING
# ------------------------------------------------------------

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# ------------------------------------------------------------
# 20. CATEGORICAL PREPROCESSING
# ------------------------------------------------------------

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


# ------------------------------------------------------------
# 21. COMBINE PREPROCESSING
# ------------------------------------------------------------

transformers = []

if len(numeric_features) > 0:

    transformers.append(
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        )
    )


if len(categorical_features) > 0:

    transformers.append(
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    )


preprocessor = ColumnTransformer(
    transformers=transformers
)


# ------------------------------------------------------------
# 22. CREATE DECISION TREE
# ------------------------------------------------------------

decision_tree = DecisionTreeClassifier(
    criterion="gini",
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)


# ------------------------------------------------------------
# 23. COMPLETE PIPELINE
# ------------------------------------------------------------

pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            decision_tree
        )
    ]
)


# ------------------------------------------------------------
# 24. TRAIN MODEL
# ------------------------------------------------------------

print(
    "\nTraining Decision Tree Classifier..."
)

pipeline.fit(
    X_train,
    y_train
)

print(
    "Model training completed!"
)


# ------------------------------------------------------------
# 25. MAKE PREDICTIONS
# ------------------------------------------------------------

y_pred = pipeline.predict(
    X_test
)

y_proba = pipeline.predict_proba(
    X_test
)


# ------------------------------------------------------------
# 26. MODEL EVALUATION
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    average="binary",
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    average="binary",
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    average="binary",
    zero_division=0
)


print("\n========== MODEL PERFORMANCE ==========")

print(
    f"Accuracy : {accuracy:.4f}"
)

print(
    f"Precision: {precision:.4f}"
)

print(
    f"Recall   : {recall:.4f}"
)

print(
    f"F1 Score : {f1:.4f}"
)


# ------------------------------------------------------------
# 27. ROC-AUC
# ------------------------------------------------------------

try:

    roc_auc = roc_auc_score(
        y_test,
        y_proba[:, 1]
    )

    print(
        f"ROC-AUC  : {roc_auc:.4f}"
    )

except Exception as e:

    print(
        "ROC-AUC could not be calculated:",
        e
    )


# ------------------------------------------------------------
# 28. CLASSIFICATION REPORT
# ------------------------------------------------------------

print(
    "\n========== CLASSIFICATION REPORT =========="
)

print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)


# ------------------------------------------------------------
# 29. CONFUSION MATRIX
# ------------------------------------------------------------

cm = confusion_matrix(
    y_test,
    y_pred
)

plt.figure(figsize=(7, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title(
    "Heart Disease Confusion Matrix"
)

plt.xlabel(
    "Predicted Class"
)

plt.ylabel(
    "Actual Class"
)

plt.show()


# ------------------------------------------------------------
# 30. ROC CURVE
# ------------------------------------------------------------

try:

    fpr, tpr, thresholds = roc_curve(
        y_test,
        y_proba[:, 1]
    )

    plt.figure(figsize=(8, 6))

    plt.plot(
        fpr,
        tpr,
        label=f"Decision Tree (AUC = {roc_auc:.3f})"
    )

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--"
    )

    plt.title(
        "ROC Curve - Decision Tree"
    )

    plt.xlabel(
        "False Positive Rate"
    )

    plt.ylabel(
        "True Positive Rate"
    )

    plt.legend()

    plt.show()

except Exception as e:

    print(
        "ROC curve could not be created:",
        e
    )


# ------------------------------------------------------------
# 31. GET PROCESSED FEATURE NAMES
# ------------------------------------------------------------

feature_names = (
    pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)


print(
    "\nNumber of processed features:",
    len(feature_names)
)


# ------------------------------------------------------------
# 32. FEATURE IMPORTANCE
# ------------------------------------------------------------

tree_model = (
    pipeline
    .named_steps["model"]
)

importance_values = (
    tree_model.feature_importances_
)


feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance_values
})


feature_importance_df = (
    feature_importance_df
    .sort_values(
        by="Importance",
        ascending=False
    )
)


print(
    "\n========== FEATURE IMPORTANCE =========="
)

display(
    feature_importance_df.head(20)
)


# ------------------------------------------------------------
# 33. FEATURE IMPORTANCE PLOT
# ------------------------------------------------------------

top_features = (
    feature_importance_df
    .head(15)
    .sort_values(
        by="Importance"
    )
)


plt.figure(figsize=(10, 7))

sns.barplot(
    data=top_features,
    x="Importance",
    y="Feature"
)

plt.title(
    "Top Features - Decision Tree"
)

plt.xlabel(
    "Importance"
)

plt.ylabel(
    "Feature"
)

plt.tight_layout()

plt.show()


# ------------------------------------------------------------
# 34. VISUALIZE DECISION TREE
# ------------------------------------------------------------

plt.figure(figsize=(25, 15))

plot_tree(
    tree_model,
    feature_names=feature_names,
    class_names=[
        str(c)
        for c in tree_model.classes_
    ],
    filled=True,
    rounded=True,
    fontsize=9
)

plt.title(
    "Decision Tree Structure"
)

plt.show()


# ------------------------------------------------------------
# 35. TREE DEPTH AND LEAVES
# ------------------------------------------------------------

print(
    "\n========== TREE INFORMATION =========="
)

print(
    "Tree depth:",
    tree_model.get_depth()
)

print(
    "Number of leaves:",
    tree_model.get_n_leaves()
)


# ------------------------------------------------------------
# 36. ACTUAL VS PREDICTED
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})


print(
    "\n========== ACTUAL VS PREDICTED =========="
)

display(
    comparison.head(20)
)


# ------------------------------------------------------------
# 37. PREDICT A NEW PATIENT
# ------------------------------------------------------------

# Use an existing patient as a template.
# This automatically matches your dataset columns.

new_patient = X_test.iloc[[0]].copy()


print(
    "\n========== SAMPLE PATIENT =========="
)

display(
    new_patient
)


new_prediction = pipeline.predict(
    new_patient
)

new_probability = pipeline.predict_proba(
    new_patient
)


print(
    "\n========== PREDICTION =========="
)

print(
    "Predicted Heart Disease Class:",
    new_prediction[0]
)


print(
    "\nPrediction probabilities:"
)

for class_label, probability in zip(
    tree_model.classes_,
    new_probability[0]
):

    print(
        f"Class {class_label}: "
        f"{probability:.4f}"
    )


# ------------------------------------------------------------
# 38. SAVE PREDICTIONS
# ------------------------------------------------------------

prediction_results = X_test.copy()

prediction_results["Actual"] = y_test.values

prediction_results["Predicted"] = y_pred


prediction_file = (
    "Heart_Disease_Decision_Tree_Predictions.csv"
)


prediction_results.to_csv(
    prediction_file,
    index=False
)


print(
    "\nPrediction file saved as:"
)

print(
    prediction_file
)


# ------------------------------------------------------------
# 39. DOWNLOAD PREDICTIONS
# ------------------------------------------------------------

files.download(
    prediction_file
)


# ------------------------------------------------------------
# 40. FINAL PROJECT SUMMARY
# ------------------------------------------------------------

print("\n")

print("=" * 60)

print(
    "PROJECT 8 - FINAL SUMMARY"
)

print("=" * 60)

print(
    "Dataset shape:",
    df.shape
)

print(
    "\nTarget variable:",
    TARGET
)

print(
    "\nModel:"
)

print(
    "Decision Tree Classifier"
)

print(
    "\nPreprocessing:"
)

print(
    "- Missing value imputation"
)

print(
    "- Numerical feature scaling"
)

print(
    "- Categorical feature encoding"
)

print(
    "\nEvaluation Metrics:"
)

print(
    f"Accuracy : {accuracy:.4f}"
)

print(
    f"Precision: {precision:.4f}"
)

print(
    f"Recall   : {recall:.4f}"
)

print(
    f"F1 Score : {f1:.4f}"
)

try:

    print(
        f"ROC-AUC  : {roc_auc:.4f}"
    )

except:

    pass


print(
    "\nTree depth:",
    tree_model.get_depth()
)

print(
    "Number of leaves:",
    tree_model.get_n_leaves()
)

print(
    "\nThe Decision Tree Classifier predicts"
)

print(
    "whether a patient has heart disease"
)

print(
    "based on the available patient features."
)

print("=" * 60)